# GraphRAG 与混合检索 — 第一节

**核心主题：**

> 纯向量检索的局限性，以及 BM25 关键词检索如何弥补这些不足

---

### 本节三个经典失败场景：

| 场景 | 问题描述 | 向量检索表现 |
|------|---------|------------|
| ① 精确关键词匹配 | 产品型号（如 WH-1000XM5） | ❌ 语义压缩丢失罕见词元 |
| ② 多跳推理 | "苹果 CEO 在哪个城市读大学？" | ❌ 无法跨文档串联关系 |
| ③ 数值/日期精度 | "April 4, 1975" | ❌ 嵌入模糊化数值细节 |

---

**安装依赖：**
```bash
pip install chromadb sentence-transformers rank-bm25
```

## 第一步 — 导入依赖

In [12]:
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi

## 第二步 — 文档语料库

构造三类文档，分别对应三个失败场景：
- **产品型号**：包含精确型号字符串
- **多跳推理**：人物 → 公司 → 大学 → 城市
- **数值/日期精度**：精确成立日期

In [13]:
DOCUMENTS = [
    # Product model numbers
    "The iPhone 15 Pro Max features a titanium frame and USB-C connector.",
    "Apple released the MacBook Pro M3 with 18-hour battery life.",
    "Samsung Galaxy S24 Ultra has a built-in S Pen stylus.",
    "The ThinkPad X1 Carbon Gen 12 weighs only 1.12 kg.",
    "Sony WH-1000XM5 headphones offer best-in-class noise cancellation.",
    # Multi-hop reasoning
    "Tim Cook is the CEO of Apple Inc., headquartered in Cupertino, California.",
    "Sundar Pichai leads Google, which is based in Mountain View, California.",
    "Satya Nadella is the CEO of Microsoft, located in Redmond, Washington.",
    "Tim Cook studied at Auburn University in Auburn, Alabama.",
    "Sundar Pichai attended Stanford University in Stanford, California.",
    # Numerical / date precision
    "Apple was founded on April 1, 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne.",
    "Google was incorporated on September 4, 1998.",
    "Microsoft was founded on April 4, 1975 by Bill Gates and Paul Allen.",
    "Apple's market cap exceeded $3 trillion for the first time in January 2022.",
    "The first iPhone was announced on January 9, 2007.",
]

DOC_IDS = [f"doc_{i}" for i in range(len(DOCUMENTS))]

print(f"Corpus: {len(DOCUMENTS)} documents")
for i, doc in enumerate(DOCUMENTS):
    print(f"  [{DOC_IDS[i]}] {doc}")

Corpus: 15 documents
  [doc_0] The iPhone 15 Pro Max features a titanium frame and USB-C connector.
  [doc_1] Apple released the MacBook Pro M3 with 18-hour battery life.
  [doc_2] Samsung Galaxy S24 Ultra has a built-in S Pen stylus.
  [doc_3] The ThinkPad X1 Carbon Gen 12 weighs only 1.12 kg.
  [doc_4] Sony WH-1000XM5 headphones offer best-in-class noise cancellation.
  [doc_5] Tim Cook is the CEO of Apple Inc., headquartered in Cupertino, California.
  [doc_6] Sundar Pichai leads Google, which is based in Mountain View, California.
  [doc_7] Satya Nadella is the CEO of Microsoft, located in Redmond, Washington.
  [doc_8] Tim Cook studied at Auburn University in Auburn, Alabama.
  [doc_9] Sundar Pichai attended Stanford University in Stanford, California.
  [doc_10] Apple was founded on April 1, 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne.
  [doc_11] Google was incorporated on September 4, 1998.
  [doc_12] Microsoft was founded on April 4, 1975 by Bill Gates and Paul Allen.
 

## 第三步 — 构建向量索引

使用 **ChromaDB**（内存模式）+ **MiniLM** 嵌入模型：

- `all-MiniLM-L6-v2`：轻量快速，384 维向量
- ChromaDB：开箱即用的向量数据库，支持余弦相似度检索

In [14]:
def build_vector_store() -> chromadb.Collection:
    """Create an in-memory ChromaDB collection with MiniLM embeddings."""
    client = chromadb.Client()
    ef = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
    try:
        client.delete_collection("demo")
    except Exception:
        pass
    collection = client.create_collection(name="demo", embedding_function=ef)
    collection.add(documents=DOCUMENTS, ids=DOC_IDS)
    return collection


print("Building vector index (downloads MiniLM on first run)…")
collection = build_vector_store()
print(f"✅ ChromaDB ready — {collection.count()} documents indexed")

Building vector index (downloads MiniLM on first run)…
✅ ChromaDB ready — 15 documents indexed


## 第四步 — 构建 BM25 索引

**BM25（Best Match 25）** 是经典的 TF-IDF 变体：
- 基于词频（TF）和逆文档频率（IDF）对文档打分
- 对**精确词元匹配**极为敏感
- 无需神经网络，纯统计模型

In [15]:
def build_bm25_index() -> BM25Okapi:
    """Tokenise documents and build a BM25 index."""
    tokenised = [doc.lower().split() for doc in DOCUMENTS]
    return BM25Okapi(tokenised)


bm25 = build_bm25_index()
print("✅ BM25 index ready")

# Tokenisation example
sample = DOCUMENTS[4].lower().split()
print(f"\nTokens for doc_4: {sample}")

✅ BM25 index ready

Tokens for doc_4: ['sony', 'wh-1000xm5', 'headphones', 'offer', 'best-in-class', 'noise', 'cancellation.']


## 第五步 — 检索工具函数

In [16]:
def vector_search(collection: chromadb.Collection, query: str, n: int = 3) -> list[str]:
    """Return top-n documents via vector similarity."""
    results = collection.query(query_texts=[query], n_results=n)
    return results["documents"][0]


def bm25_search(index: BM25Okapi, query: str, n: int = 3) -> list[str]:
    """Return top-n documents via BM25 keyword scoring."""
    tokens = query.lower().split()
    scores = index.get_scores(tokens)
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [DOCUMENTS[i] for i in ranked[:n]]


def print_results(label: str, results: list[str]) -> None:
    print(f"  [{label}]")
    for i, doc in enumerate(results, 1):
        print(f"    {i}. {doc}")


print("✅ Search functions ready")

✅ Search functions ready


---
## 🔍 场景一：精确关键词匹配

**查询**：`WH-1000XM5`（索尼耳机型号）

**向量检索为何失败**：
- 嵌入将语义压缩为 384 维浮点数
- 罕见型号字符串在训练数据中几乎不出现
- `WH-1000XM5` 在嵌入空间中可能距 `headphones` 很远

**期望结果**：`Sony WH-1000XM5 headphones offer best-in-class noise cancellation.`

In [17]:
query_1 = "WH-1000XM5"
print(f"Query: '{query_1}'")
print("  Expected: Sony WH-1000XM5 headphones doc\n")

print_results("Vector Search", vector_search(collection, query_1))
print()
print_results("BM25 Search  ", bm25_search(bm25, query_1))

Query: 'WH-1000XM5'
  Expected: Sony WH-1000XM5 headphones doc

  [Vector Search]
    1. Sony WH-1000XM5 headphones offer best-in-class noise cancellation.
    2. The ThinkPad X1 Carbon Gen 12 weighs only 1.12 kg.
    3. Apple released the MacBook Pro M3 with 18-hour battery life.

  [BM25 Search  ]
    1. Sony WH-1000XM5 headphones offer best-in-class noise cancellation.
    2. The iPhone 15 Pro Max features a titanium frame and USB-C connector.
    3. Apple released the MacBook Pro M3 with 18-hour battery life.


---
## 🔗 场景二：多跳推理

**查询**：`Which city did the CEO of Apple attend university in?`

**推理链（需要多跳）**：
```
Apple CEO → Tim Cook → Auburn University → Auburn, Alabama
```

**两种方法均失败的原因**：
- 单次检索只能召回语义最相近的文档
- 两种方法都无法跨文档建立**关系链**
- 这正是 **GraphRAG** 要解决的核心问题！

**期望**：需同时召回 CEO 文档 + 大学文档并关联，才能推理出 *Auburn, Alabama*

In [18]:
query_2 = "Which city did the CEO of Apple attend university in?"
print(f"Query: '{query_2}'")
print("  Expected: Auburn, Alabama  (Tim Cook → Auburn University → Auburn, AL)\n")

print_results("Vector Search", vector_search(collection, query_2))
print()
print_results("BM25 Search  ", bm25_search(bm25, query_2))

Query: 'Which city did the CEO of Apple attend university in?'
  Expected: Auburn, Alabama  (Tim Cook → Auburn University → Auburn, AL)

  [Vector Search]
    1. Tim Cook is the CEO of Apple Inc., headquartered in Cupertino, California.
    2. Apple was founded on April 1, 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne.
    3. Apple's market cap exceeded $3 trillion for the first time in January 2022.

  [BM25 Search  ]
    1. Tim Cook is the CEO of Apple Inc., headquartered in Cupertino, California.
    2. Satya Nadella is the CEO of Microsoft, located in Redmond, Washington.
    3. Sundar Pichai leads Google, which is based in Mountain View, California.


---
## 📅 场景三：数值/日期精度

**查询**：`April 4 1975`

**向量检索为何失败**：
- 嵌入模型将数字视为连续语义，而非精确符号
- `1975` 与 `1976` 在嵌入空间中非常接近（相邻年份）
- 日期格式微小变化（`April 4` vs `4th of April`）也会偏移向量

**期望结果**：`Microsoft was founded on April 4, 1975 by Bill Gates and Paul Allen.`

In [19]:
query_3 = "April 4 1975"
print(f"Query: '{query_3}'")
print("  Expected: Microsoft founded on April 4, 1975\n")

print_results("Vector Search", vector_search(collection, query_3))
print()
print_results("BM25 Search  ", bm25_search(bm25, query_3))

Query: 'April 4 1975'
  Expected: Microsoft founded on April 4, 1975

  [Vector Search]
    1. Microsoft was founded on April 4, 1975 by Bill Gates and Paul Allen.
    2. Google was incorporated on September 4, 1998.
    3. Apple was founded on April 1, 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne.

  [BM25 Search  ]
    1. Microsoft was founded on April 4, 1975 by Bill Gates and Paul Allen.
    2. Apple was founded on April 1, 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne.
    3. The iPhone 15 Pro Max features a titanium frame and USB-C connector.


---
## 📊 总结与结论

| 检索方式 | 优势 | 劣势 |
|---------|------|------|
| **向量检索** | 语义相似，处理同义词/改写 | 精确词元匹配差，数值模糊 |
| **BM25** | 精确词元匹配，无需训练 | 无语义理解，无法跨文档推理 |
| **混合检索（向量 + BM25）** | 兼顾语义与精确匹配 | 仍无法解决多跳推理 |
| **GraphRAG** | 图遍历实现多跳推理 | 构建成本高，需要图数据库 |

### 核心洞察

```
向量检索  → 语义相似性强，精确匹配与多跳推理弱
BM25      → 精确匹配强，语义理解盲区
混合检索  → 互补，覆盖更多场景
GraphRAG  → 通过图遍历解决多跳推理的终极方案
```

**下一节：实现混合检索（Vector + BM25 + RRF 融合），并引入图结构解决多跳推理问题。**

In [20]:
print("=" * 60)
print("核心结论")
print("=" * 60)
print("  向量检索擅长语义相似，但在以下场景表现不佳：")
print("  • 精确词元匹配（产品型号、编码）")
print("  • 多跳推理（需要图结构遍历）")
print("  • 数值/日期精度（嵌入压缩了数值细节）")
print()
print("  BM25 擅长精确匹配，但无法理解语义意图。")
print()
print("  → 混合检索 + 图结构遍历 = 兼顾所有场景的最优方案")

TAKEAWAY
  Vector search excels at semantic similarity but struggles with:
  • Exact token matches (model numbers, codes)
  • Multi-hop reasoning (requires graph traversal)
  • Numerical/date precision (embeddings compress numerics)

  BM25 handles exact matches well but misses semantic intent.

  → Hybrid retrieval + Graph traversal gives the best of all worlds.
